<a href="https://colab.research.google.com/github/thinhlpg/vixtts-demo/blob/dev/viXTTS_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔥🔥🔥**viXTTS Demo**🗣️🗣️🗣️

- https://github.com/thinhlpg/vixtts-demo/
- https://github.com/thinhlpg/viVoice

Demo này giúp bạn chạy viXTTS miễn phí trên Google Colab!
Xem thông tin mô hình tại [đây](https://huggingface.co/capleaf/viXTTS)

Bạn có thể dùng demo này với mục đích:
- ✅Mục đích cá nhân, học tập, nghiên cứu, thử nghiệm

Bạn **KHÔNG** được dùng demo này với mục đích:
- ❌Mục đích trái đạo đức, vi phạm pháp luật Việt Nam
- ❌Tạo ra nội dung gây thù ghét, kỳ thị, bạo lực hoặc nội dung vi phạm bản quyền
- ❌Giả mạo danh tính hoặc gây hiểu nhầm rằng nội dung được tạo ra bởi một cá nhân hoặc tổ chức khác

In [ ]:
# @title 1. ⚙️ **Cài đặt**
# @markdown 👈Nhấn nút này để cài đặt (~5 phút)
# Change timezone to Vietnam
!rm /etc/localtime
!ln -s /usr/share/zoneinfo/Asia/Ho_Chi_Minh /etc/localtime
!date

print(" > Cài đặt thư viện...")
!rm -rf TTS/
!git clone --branch add-vietnamese-xtts -q https://github.com/thinhlpg/TTS.git
!pip install --use-deprecated=legacy-resolver -q -e TTS
!pip install deepspeed -q
!pip install -q vinorm==2.0.7
!pip install -q cutlet
!pip install -q unidic==1.1.0
!pip install -q underthesea
!pip install -q gradio==4.35
!pip install deepfilternet==0.5.6 -q

import os
from huggingface_hub import snapshot_download


os.system("python -m unidic download")
print(" > Tải mô hình...")
snapshot_download(repo_id="thinhlpg/viXTTS",
                  repo_type="model",
                  local_dir="model")

from IPython.display import clear_output
clear_output()
print(" > ✅ Cài đặt hoàn tất, bạn hãy chạy tiếp các bước tiếp theo nhé!")
quit()

In [ ]:
!pip install --use-deprecated=legacy-resolver -q -e TTS
!pip install deepspeed -q
!pip install -q vinorm==2.0.7
!pip install -q cutlet
!pip install -q unidic==1.1.0
!pip install -q underthesea
!pip install -q gradio==4.35
!pip install deepfilternet==0.5.6 -q

In [ ]:
import torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
import nltk
from pathlib import Path
import numpy as np
from typing import List, Optional
# The inference code is adopted from https://github.com/coqui-ai/TTS/blob/dev/TTS/demos/xtts_ft_demo/xtts_demo.py
# @title 2. 🤗 **Sử dụng**
# @markdown 👈 Nhấn để chạy.
# @markdown Nếu gặp lỗi thì cũng nhấn nút này nhé!

# @markdown Lần đầu chạy sẽ hơi lâu, bạn chờ tí nhé!

# @markdown Kết quả sẽ được lưu vào `/content/output`

# @markdown Chọn ngôn ngữ:
language = "Tiếng Việt" # @param ["Tiếng Việt", "Tiếng Anh","Tiếng Tây Ban Nha", "Tiếng Pháp","Tiếng Đức","Tiếng Ý", "Tiếng Bồ Đào Nha", "Tiếng Ba Lan", "Tiếng Thổ Nhĩ Kỳ", "Tiếng Nga", "Tiếng Hà Lan", "Tiếng Séc", "Tiếng Ả Rập", "Tiếng Trung (giản thể)", "Tiếng Nhật", "Tiếng Hungary", "Tiếng Hàn", "Tiếng Hindi"]
# @markdown Văn bản để đọc. Độ dài tối thiểu mỗi câu nên từ 10 từ để đặt kết quả tốt nhất.
input_text ="""Sáng 10/2, tại Trụ sở Chính phủ, Thủ tướng Phạm Minh Chính chủ trì Hội nghị Thường trực Chính phủ gặp gỡ doanh nghiệp về nhiệm vụ, giải pháp để doanh nghiệp tư nhân tăng tốc, bứt phá, góp phần phát triển đất nước nhanh, bền vững trong kỷ nguyên mới.
Cùng dự, có các Phó Thủ tướng Chính phủ Nguyễn Hòa Bình, Trần Hồng Hà, Lê Thành Long và Bùi Thanh Sơn; lãnh đạo các bộ, ngành Trung ương, các hiệp hội doanh nghiệp, 26 doanh nghiệp nhà nước và tư nhân lớn.
Phát biểu ý kiến khai mạc hội nghị, thay mặt Tổng Bí thư Tô Lâm, các đồng chí Lãnh đạo Đảng, Nhà nước, Thủ tướng Phạm Minh Chính gửi tới các doanh nghiệp lời chào trân trọng, lời thăm hỏi ân cần, lời chúc mừng tốt đẹp nhất; nêu rõ, chúng ta đã bước sang năm cuối của nhiệm kỳ Đại hội lần thứ XIII của Đảng, đây là giai đoạn khó khăn với đại dịch Covid-19 hoành hành; chiến tranh, xung đột trên thế giới làm đứt gãy chuỗi cung ứng; cơn bão số 3 (Yagi) gây hậu quả nghiêm trọng; trong năm 2024 có sự thay đổi nhân sự lãnh đạo cấp cao, sự ra đi đột ngột của Tổng Bí thư Nguyễn Phú Trọng... tác động đến tình hình kinh tế-xã hội, an ninh chính trị, quốc phòng… đất nước nhưng dưới sự lãnh đạo của Đảng, thường xuyên trực tiếp là Bộ Chính trị, Ban Bí thư, đứng đầu là đồng chí Tổng Bí thư, cả hệ thống chính trị vào cuộc, có sự ủng hộ, đồng tình của người dân, doanh nghiệp, sự giúp đỡ của bạn bè quốc tế, chúng ta đã nỗ lực vượt qua mọi khó khăn khốc liệt, đạt được những thành tựu phát triển kinh tế-xã hội ấn tượng.
""" 



# @param {type:"string"}
# @markdown Chọn giọng mẫu:
reference_audio = "2_20231227_1742037149.wav" # @param [ "model/user_sample.wav",  "model/vi_sample.wav",  "model/samples/nam-calm.wav",  "model/samples/nam-cham.wav",  "model/samples/nam-nhanh.wav",  "model/samples/nam-truyen-cam.wav",  "model/samples/nu-calm.wav",  "model/samples/nu-cham.wav",  "model/samples/nu-luu-loat.wav",  "model/samples/nu-nhan-nha.wav",  "model/samples/nu-nhe-nhang.wav"]
# @markdown Tự động chuẩn hóa chữ (VD: 20/11 -> hai mươi tháng mười một)
normalize_text = True # @param {type:"boolean"}
# @markdown In chi tiết xử lý
verbose = True # @param {type:"boolean"}
# @markdown Lưu từng câu thành file riêng lẻ.
output_chunks = True # @param {type:"boolean"}

from IPython.display import clear_output
def cry_and_quit():
    clear_output()
    print("> Lỗi rồi huhu 😭😭, bạn hãy nhấn chạy lại phần này nhé!")
    quit()

import os
import string
import unicodedata
from datetime import datetime
from pprint import pprint

import torch
import torchaudio
from tqdm import tqdm
from underthesea import sent_tokenize
from unidecode import unidecode

try:
    from vinorm import TTSnorm
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts
except:
    cry_and_quit()

# Load model
def clear_gpu_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
class XTTSGenerator:
    def __init__(self, model_path: str, xtts_config: str, xtts_vocab: str):
        """
        Initialize XTTS generator with model path.
        
        Args:
            model_path: Path to the XTTS model directory
        """
        self.config = XttsConfig()
        self.config.load_json(xtts_config)
        self.model = Xtts.init_from_config(self.config)
        self.model.load_checkpoint(config,
                                   checkpoint_path=xtts_checkpoint,
                                   vocab_path=xtts_vocab,
                                   use_deepspeed=True)
        if torch.cuda.is_available():
            self.model.cuda()
        # self.model.cuda()
        
        # Initialize NLTK for sentence tokenization
        # try:
        #     nltk.data.find('tokenizers/punkt')
        # except LookupError:
        #     nltk.download('punkt')

    def load_model(xtts_checkpoint, xtts_config, xtts_vocab):
        clear_gpu_cache()
        if not xtts_checkpoint or not xtts_config or not xtts_vocab:
            return "You need to run the previous steps or manually set the `XTTS checkpoint path`, `XTTS config path`, and `XTTS vocab path` fields !!"
        config = XttsConfig()
        config.load_json(xtts_config)
        XTTS_MODEL = Xtts.init_from_config(config)
        print("Loading XTTS model! ")
        XTTS_MODEL.load_checkpoint(config,
                                   checkpoint_path=xtts_checkpoint,
                                   vocab_path=xtts_vocab,
                                   use_deepspeed=True)
        if torch.cuda.is_available():
            XTTS_MODEL.cuda()
    
        print("Model Loaded!")
        return XTTS_MODEL
    
    def split_text(self, text: str, max_tokens: int = 400) -> List[str]:
        """
        Split text into chunks that respect the token limit.
        
        Args:
            text: Input text to split
            max_tokens: Maximum tokens per chunk (default: 400)
            
        Returns:
            List of text chunks
        """
        sentences = nltk.sent_tokenize(text)
        chunks = []
        current_chunk = []
        current_length = 0
        
        for sentence in sentences:
            # Approximate token count (words + punctuation)
            sentence_tokens = len(sentence)
            
            if current_length + sentence_tokens > max_tokens:
                if current_chunk:
                    chunks.append(' '.join(current_chunk))
                current_chunk = [sentence]
                current_length = sentence_tokens
            else:
                current_chunk.append(sentence)
                current_length += sentence_tokens
        
        if current_chunk:
            chunks.append(' '.join(current_chunk))
            
        return chunks

    def normalize_vietnamese_text(text):
        text = (
            TTSnorm(text, unknown=False, lower=False, rule=True)
            .replace("..", ".")
            .replace("!.", "!")
            .replace("?.", "?")
            .replace(" .", ".")
            .replace(" ,", ",")
            .replace('"', "")
            .replace("'", "")
            .replace("AI", "Ây Ai")
            .replace("A.I", "Ây Ai")
        )
        return text
    
    def generate_speech(
        self,
        text: str,
        speaker_wav: str,
        language: str = "en",
        output_path: Optional[str] = None
    ) -> np.ndarray:
        """
        Generate speech from text using XTTS.
        
        Args:
            text: Input text to convert to speech
            speaker_wav: Path to reference speaker audio file
            language: Language code (default: "en")
            output_path: Optional path to save audio file
            
        Returns:
            numpy array containing the generated audio
        """
        chunks = self.split_text(text)
        full_audio = []
        
        for chunk in chunks:
            with torch.no_grad():
                gpt_cond_latents, speaker_embedding = self.model.get_conditioning_latents(
                    audio_path=speaker_wav,
                    gpt_cond_len=len(chunk),
                    max_ref_length=10
                )

                if normalize_text and lang == "vi":
                    try:
                        tts_text = self.normalize_vietnamese_text(chunk)
                    except:
                        cry_and_quit()
        
                if lang in ["ja", "zh-cn"]:
                    tts_texts = tts_text.split("。")
                else:
                    tts_texts = sent_tokenize(tts_text)
        
                chunk_audio = self.model.inference(
                    text=chunk,
                    language=language,
                    gpt_cond_latents=gpt_cond_latents,
                    speaker_embedding=speaker_embedding,
                    temperature=0.7
                )
                
                full_audio.append(chunk_audio)
        
        # Concatenate all audio chunks
        final_audio = np.concatenate(full_audio)
        
        # Save audio if output path is provided
        if output_path:
            self.model.save_wav(final_audio, output_path)
            
        return final_audio

In [ ]:
# The inference code is adopted from https://github.com/coqui-ai/TTS/blob/dev/TTS/demos/xtts_ft_demo/xtts_demo.py
# @title 2. 🤗 **Sử dụng**
# @markdown 👈 Nhấn để chạy.
# @markdown Nếu gặp lỗi thì cũng nhấn nút này nhé!

# @markdown Lần đầu chạy sẽ hơi lâu, bạn chờ tí nhé!

# @markdown Kết quả sẽ được lưu vào `/content/output`

# @markdown Chọn ngôn ngữ:
language = "Tiếng Việt" # @param ["Tiếng Việt", "Tiếng Anh","Tiếng Tây Ban Nha", "Tiếng Pháp","Tiếng Đức","Tiếng Ý", "Tiếng Bồ Đào Nha", "Tiếng Ba Lan", "Tiếng Thổ Nhĩ Kỳ", "Tiếng Nga", "Tiếng Hà Lan", "Tiếng Séc", "Tiếng Ả Rập", "Tiếng Trung (giản thể)", "Tiếng Nhật", "Tiếng Hungary", "Tiếng Hàn", "Tiếng Hindi"]
# @markdown Văn bản để đọc. Độ dài tối thiểu mỗi câu nên từ 10 từ để đặt kết quả tốt nhất.
input_text ="""Sáng 10/2, tại Trụ sở Chính phủ, Thủ tướng Phạm Minh Chính chủ trì Hội nghị Thường trực Chính phủ gặp gỡ doanh nghiệp về nhiệm vụ, giải pháp để doanh nghiệp tư nhân tăng tốc, bứt phá, góp phần phát triển đất nước nhanh, bền vững trong kỷ nguyên mới.
Cùng dự, có các Phó Thủ tướng Chính phủ Nguyễn Hòa Bình, Trần Hồng Hà, Lê Thành Long và Bùi Thanh Sơn; lãnh đạo các bộ, ngành Trung ương, các hiệp hội doanh nghiệp, 26 doanh nghiệp nhà nước và tư nhân lớn.
Phát biểu ý kiến khai mạc hội nghị, thay mặt Tổng Bí thư Tô Lâm, các đồng chí Lãnh đạo Đảng, Nhà nước. Thủ tướng Phạm Minh Chính gửi tới các doanh nghiệp lời chào trân trọng, lời thăm hỏi ân cần, lời chúc mừng tốt đẹp nhất; nêu rõ, chúng ta đã bước sang năm cuối của nhiệm kỳ Đại hội lần thứ XIII của Đảng, đây là giai đoạn khó khăn với đại dịch Covid-19 hoành hành; chiến tranh, xung đột trên thế giới làm đứt gãy chuỗi cung ứng; cơn bão số 3 (Yagi) gây hậu quả nghiêm trọng. Trong năm 2024 có sự thay đổi nhân sự lãnh đạo cấp cao, sự ra đi đột ngột của Tổng Bí thư Nguyễn Phú Trọng... tác động đến tình hình kinh tế-xã hội, an ninh chính trị, quốc phòng… đất nước nhưng dưới sự lãnh đạo của Đảng, thường xuyên trực tiếp là Bộ Chính trị, Ban Bí thư, đứng đầu là đồng chí Tổng Bí thư, cả hệ thống chính trị vào cuộc, có sự ủng hộ, đồng tình của người dân, doanh nghiệp, sự giúp đỡ của bạn bè quốc tế, chúng ta đã nỗ lực vượt qua mọi khó khăn khốc liệt, đạt được những thành tựu phát triển kinh tế-xã hội ấn tượng.
""" 



# @param {type:"string"}
# @markdown Chọn giọng mẫu:
reference_audio = "2_20231227_1742037149.wav" # @param [ "model/user_sample.wav",  "model/vi_sample.wav",  "model/samples/nam-calm.wav",  "model/samples/nam-cham.wav",  "model/samples/nam-nhanh.wav",  "model/samples/nam-truyen-cam.wav",  "model/samples/nu-calm.wav",  "model/samples/nu-cham.wav",  "model/samples/nu-luu-loat.wav",  "model/samples/nu-nhan-nha.wav",  "model/samples/nu-nhe-nhang.wav"]
# @markdown Tự động chuẩn hóa chữ (VD: 20/11 -> hai mươi tháng mười một)
normalize_text = True # @param {type:"boolean"}
# @markdown In chi tiết xử lý
verbose = True # @param {type:"boolean"}
# @markdown Lưu từng câu thành file riêng lẻ.
output_chunks = True # @param {type:"boolean"}

from IPython.display import clear_output
def cry_and_quit():
    clear_output()
    print("> Lỗi rồi huhu 😭😭, bạn hãy nhấn chạy lại phần này nhé!")
    quit()

import os
import string
import unicodedata
from datetime import datetime
from pprint import pprint

import torch
import torchaudio
from tqdm import tqdm
from underthesea import sent_tokenize
from unidecode import unidecode

try:
    from vinorm import TTSnorm
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts
except:
    cry_and_quit()

# Load model
def clear_gpu_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model(xtts_checkpoint, xtts_config, xtts_vocab):
    clear_gpu_cache()
    if not xtts_checkpoint or not xtts_config or not xtts_vocab:
        return "You need to run the previous steps or manually set the `XTTS checkpoint path`, `XTTS config path`, and `XTTS vocab path` fields !!"
    config = XttsConfig()
    config.load_json(xtts_config)
    XTTS_MODEL = Xtts.init_from_config(config)
    print("Loading XTTS model! ")
    XTTS_MODEL.load_checkpoint(config,
                               checkpoint_path=xtts_checkpoint,
                               vocab_path=xtts_vocab,
                               use_deepspeed=True)
    if torch.cuda.is_available():
        XTTS_MODEL.cuda()

    print("Model Loaded!")
    return XTTS_MODEL


def get_file_name(text, max_char=50):
    filename = text[:max_char]
    filename = filename.lower()
    filename = filename.replace(" ", "_")
    filename = filename.translate(str.maketrans("", "", string.punctuation.replace("_", "")))
    filename = unidecode(filename)
    current_datetime = datetime.now().strftime("%m%d%H%M%S")
    filename = f"{current_datetime}_{filename}"
    return filename


def calculate_keep_len(text, lang):
    if lang in ["ja", "zh-cn"]:
        return -1

    word_count = len(text.split())
    num_punct = (
        text.count(".")
        + text.count("!")
        + text.count("?")
        + text.count(",")
    )

    if word_count < 5:
        return 15000 * word_count + 2000 * num_punct
    elif word_count < 10:
        return 13000 * word_count + 2000 * num_punct
    return -1


def normalize_vietnamese_text(text):
    text = (
        TTSnorm(text, unknown=False, lower=False, rule=True)
        .replace("..", ".")
        .replace("!.", "!")
        .replace("?.", "?")
        .replace(" .", ".")
        .replace(" ,", ",")
        .replace('"', "")
        .replace("'", "")
        .replace("AI", "Ây Ai")
        .replace("A.I", "Ây Ai")
    )
    return text

def split_text(tts_text, lang='vi', max_tokens = 250):
        if lang in ["ja", "zh-cn"]:
            sentences = tts_text.split("。")
        else:
            ss = sent_tokenize(tts_text)
            sentences = []
            for s in ss:
                sentences.extend(s.split(";"))

        print("AHXHXJXHJXHJX", len(sentences))
        chunks = []
        current_chunk = []
        current_length = 0
        
        for sentence in sentences:
            # Approximate token count (words + punctuation)
            sentence_tokens = len(sentence)
            
            if current_length + sentence_tokens > max_tokens:
                # if current_chunk:
                #     chunks.append(' '.join(current_chunk))
                # current_chunk = [sentence]
                if len(current_chunk) > 0:
                    current_length = sentence_tokens
                    chunks.append(' '.join(current_chunk))
                    current_chunk = []
                    current_length = 0
                else:
                    chunks.append(sentence)
            else:
                current_chunk.append(sentence)
                current_length += sentence_tokens
        
            # if current_chunk:
            #     chunks.append(' '.join(current_chunk))
            
        return chunks
    
def run_tts(XTTS_MODEL, lang, tts_text, speaker_audio_file,
            normalize_text= True,
            verbose=False,
            output_chunks=False):
    """
    Run text-to-speech (TTS) synthesis using the provided XTTS_MODEL.

    Args:
        XTTS_MODEL: A pre-trained TTS model.
        lang (str): The language of the input text.
        tts_text (str): The text to be synthesized into speech.
        speaker_audio_file (str): Path to the audio file of the speaker to condition the synthesis on.
        normalize_text (bool, optional): Whether to normalize the input text. Defaults to True.
        verbose (bool, optional): Whether to print verbose information. Defaults to False.
        output_chunks (bool, optional): Whether to save synthesized speech chunks separately. Defaults to False.

    Returns:
        str: Path to the synthesized audio file.
    """

    if XTTS_MODEL is None or not speaker_audio_file:
        return "You need to run the previous step to load the model !!", None, None

    output_dir = "./output"
    os.makedirs(output_dir, exist_ok=True)

    gpt_cond_latent, speaker_embedding = XTTS_MODEL.get_conditioning_latents(
        audio_path=speaker_audio_file,
        gpt_cond_len=XTTS_MODEL.config.gpt_cond_len,
        max_ref_length=XTTS_MODEL.config.max_ref_len,
        sound_norm_refs=XTTS_MODEL.config.sound_norm_refs,
    )

    if normalize_text and lang == "vi":
        # Bug on google colab
        try:
            tts_text = normalize_vietnamese_text(tts_text)
        except:
            cry_and_quit()

    # if lang in ["ja", "zh-cn"]:
    #     tts_texts = tts_text.split("。")
    # else:
    #     tts_texts = sent_tokenize(tts_text)
    tts_texts = split_text(tts_text)
    
    if verbose:
        print("Text for TTS:")
        print(len(tts_texts), tts_texts)

    wav_chunks = []
    for text in tqdm(tts_texts):
        print("AJKLSJAKL", len(text))
        if text.strip() == "":
            continue

        wav_chunk = XTTS_MODEL.inference(
            text=text,
            language=lang,
            gpt_cond_latent=gpt_cond_latent,
            speaker_embedding=speaker_embedding,
            temperature=0.3,
            length_penalty=1.0,
            repetition_penalty=10.0,
            top_k=30,
            top_p=0.85,
        )

        # Quick hack for short sentences
        keep_len = calculate_keep_len(text, lang)
        wav_chunk["wav"] = torch.tensor(wav_chunk["wav"][:keep_len])

        if output_chunks:
            out_path = os.path.join(output_dir, f"{get_file_name(text)}.wav")
            torchaudio.save(out_path, wav_chunk["wav"].unsqueeze(0), 24000)
            if verbose:
                print(f"Saved chunk to {out_path}")

        wav_chunks.append(wav_chunk["wav"])

    out_wav = torch.cat(wav_chunks, dim=0).unsqueeze(0)
    out_path = os.path.join(output_dir, f"{get_file_name(tts_text)}.wav")
    torchaudio.save(out_path, out_wav, 24000)

    if verbose:
        print(f"Saved final file to {out_path}")

    return out_path


language_code_map = {
    "Tiếng Việt": "vi",
    "Tiếng Anh": "en",
    "Tiếng Trung (giản thể)": "zh-cn",
    "Tiếng Nhật": "ja",
}

print("> Đang nạp mô hình...")
try:
    if not vixtts_model:
        vixtts_model = load_model(xtts_checkpoint="model/model.pth",
                                xtts_config="model/config.json",
                                xtts_vocab="model/vocab.json")
except:
    vixtts_model = load_model(xtts_checkpoint="model/model.pth",
                                xtts_config="model/config.json",
                                xtts_vocab="model/vocab.json")
clear_output()
print("> Đã nạp mô hình")

if not os.path.exists(reference_audio):
    print("⚠️⚠️⚠️Bạn chưa tải file âm thanh lên. Hãy chọn giọng khác, hoặc tải file của bạn lên ở bên dưới.⚠️⚠️⚠️")
    audio_file="/content/model/vi_sample.wav"
else:
    audio_file = run_tts(vixtts_model,
            lang=language_code_map[language],
            tts_text=input_text,
            speaker_audio_file=reference_audio,
            normalize_text=normalize_text,
            verbose=verbose,
            output_chunks=output_chunks,)

from IPython.display import Audio
Audio(audio_file)

In [ ]:
# @title 🎤 **Tải file âm thanh của bạn lên**
# @markdown Để đạt chất lượng tốt nhất, hãy tham khảo file '/content/model/vi_sample.wav'
# @
import os
import locale
from google.colab import files

denoise = True # @param {type:"boolean"}

# Upload the audio file
uploaded = files.upload()
for filename in uploaded.keys():
    # Convert the audio file to WAV format using ffmpeg
    uploaded_dir = os.path.dirname(filename)
    if denoise:
        !deepFilter "{filename}"
        !ffmpeg -i "{filename.replace('.wav', '_DeepFilterNet3.wav')}" -ac 1 -ar 22050 -vn /content/model/user_sample.wav -y -hide_banner -loglevel error
        os.remove(filename.replace('.wav', '_DeepFilterNet3.wav'))
    else:
        !ffmpeg -i "{filename}" -ac 1 -ar 22050 -vn /content/model/user_sample.wav -y -hide_banner -loglevel error
        os.remove(filename)
    break

from IPython.display import Audio, clear_output
clear_output()
print("> Đã tải file âm thanh lên")
Audio("/content/model/user_sample.wav")

In [ ]:
# @title ⏬ **Lưu kết quả vào Drive**
# @markdown Chạy phần này để lưu kết quả vào Google Drive của bạn
# @markdown `/content/drive/MyDrive/vixtts-output`

from google.colab import drive
drive.mount('/content/drive')

# Save the output folder in "vixtts-output" in Google Drive, without overwriting existing files
!cp -n -r /content/output/* /content/drive/MyDrive/vixtts-output
print("> Đã lưu kết quả vào Google Drive")

In [ ]:
# @title ⚠️ **Dọn kết quả**
# @markdown Chạy phần này để xóa toàn bộ file trong `/content/output`
import shutil
shutil.rmtree('/content/output')
print("Đã xóa toàn bộ file trong /content/output")

In [ ]:
# @title 📴 **Tắt Demo**
# @markdown Khi chạy xong thì bạn hãy tắt demo để tiết kiệm GPU nhé!
from google.colab import runtime
runtime.unassign()

In [15]:
import os
import string
import torch
import torchaudio
from datetime import datetime
from typing import List, Optional, Tuple, Union
from tqdm import tqdm
from underthesea import sent_tokenize
from unidecode import unidecode
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from vinorm import TTSnorm

class VietnameseXTTS:
    LANGUAGE_CODE_MAP = {
        "Tiếng Việt": "vi",
        "Tiếng Anh": "en",
        "Tiếng Trung (giản thể)": "zh-cn",
        "Tiếng Nhật": "ja"
    }

    def __init__(
        self,
        model_path: str = "model/model.pth",
        config_path: str = "model/config.json",
        vocab_path: str = "model/vocab.json",
        output_dir: str = "./output"
    ):
        """
        Initialize the Vietnamese XTTS model.
        
        Args:
            model_path: Path to the model checkpoint
            config_path: Path to the model config file
            vocab_path: Path to the vocabulary file
            output_dir: Directory to save output files
        """
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.model = self._load_model(model_path, config_path, vocab_path)

    def _clear_gpu_cache(self):
        """Clear GPU cache if available."""
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _load_model(self, model_path: str, config_path: str, vocab_path: str) -> Xtts:
        """
        Load the XTTS model.
        
        Args:
            model_path: Path to model checkpoint
            config_path: Path to model config
            vocab_path: Path to vocabulary file
            
        Returns:
            Loaded XTTS model
        """
        self._clear_gpu_cache()
        
        config = XttsConfig()
        config.load_json(config_path)
        model = Xtts.init_from_config(config)
        
        print("Loading XTTS model...")
        model.load_checkpoint(
            config,
            checkpoint_path=model_path,
            vocab_path=vocab_path,
            use_deepspeed=True
        )
        
        if torch.cuda.is_available():
            model.cuda()
        
        print("Model Loaded!")
        return model

    @staticmethod
    def _get_file_name(text: str, max_char: int = 50) -> str:
        """Generate a filename from text."""
        filename = text[:max_char].lower()
        filename = filename.replace(" ", "_")
        filename = filename.translate(str.maketrans("", "", string.punctuation.replace("_", "")))
        filename = unidecode(filename)
        current_datetime = datetime.now().strftime("%m%d%H%M%S")
        return f"{current_datetime}_{filename}"

    @staticmethod
    def _calculate_keep_len(text: str, lang: str) -> int:
        """Calculate length to keep for the audio output."""
        if lang in ["ja", "zh-cn"]:
            return -1

        word_count = len(text.split())
        num_punct = sum(text.count(p) for p in ".,!?")

        if word_count < 5:
            return 15000 * word_count + 2000 * num_punct
        elif word_count < 10:
            return 13000 * word_count + 2000 * num_punct
        return -1

    @staticmethod
    def normalize_vietnamese_text(text: str) -> str:
        """Normalize Vietnamese text for TTS."""
        text = (
            TTSnorm(text, unknown=False, lower=False, rule=True)
            .replace("..", ".")
            .replace("!.", "!")
            .replace("?.", "?")
            .replace(" .", ".")
            .replace(" ,", ",")
            .replace('"', "")
            .replace("'", "")
            .replace("AI", "Ây Ai")
            .replace("A.I", "Ây Ai")
        )
        return text

    def split_text(self, text: str, lang: str = 'vi', max_tokens: int = 250) -> List[str]:
        """Split text into manageable chunks."""
        if lang in ["ja", "zh-cn"]:
            sentences = text.split("。")
        else:
            ss = sent_tokenize(text)
            g = []
            for s in ss:
                g.extend(s.split(";"))
            sentences = []
            for s in g:
                sentences.extend(s.split(","))

        chunks = []
        current_chunk = []
        current_length = 0
        
        for sentence in sentences:
            sentence_tokens = len(sentence)
            
            if current_length + sentence_tokens > max_tokens:
                if len(current_chunk) > 0:
                    chunks.append(' '.join(current_chunk))
                    current_chunk = []
                    current_length = 0
                else:
                    chunks.append(sentence)
            else:
                current_chunk.append(sentence)
                current_length += sentence_tokens
                
        if current_chunk:
            chunks.append(' '.join(current_chunk))
            
        return chunks

    def generate_speech(
        self,
        text: str,
        speaker_audio_file: str,
        language: str = "Tiếng Việt",
        normalize_text: bool = True,
        verbose: bool = False,
        output_chunks: bool = False
    ) -> str:
        """
        Generate speech from text.
        
        Args:
            text: Input text to convert to speech
            speaker_audio_file: Path to reference speaker audio
            language: Language of the input text
            normalize_text: Whether to normalize the text
            verbose: Whether to print detailed information
            output_chunks: Whether to save individual chunks
            
        Returns:
            Path to the generated audio file
        """
        lang_code = self.LANGUAGE_CODE_MAP.get(language, "vi")
        
        # Get speaker conditioning
        gpt_cond_latent, speaker_embedding = self.model.get_conditioning_latents(
            audio_path=speaker_audio_file,
            gpt_cond_len=self.model.config.gpt_cond_len,
            max_ref_length=self.model.config.max_ref_len,
            sound_norm_refs=self.model.config.sound_norm_refs,
        )

        # Normalize text if needed
        if normalize_text and lang_code == "vi":
            text = self.normalize_vietnamese_text(text)

        # Split text into chunks
        text_chunks = self.split_text(text, lang_code)
        if verbose:
            print(f"Processing {len(text_chunks)} chunks:")
            print(text_chunks)

        # Process each chunk
        wav_chunks = []
        for chunk in tqdm(text_chunks):
            print("ASJHKAHSKJA", len(chunk))
            if not chunk.strip():
                continue

            wav_chunk = self.model.inference(
                text=chunk,
                language=lang_code,
                gpt_cond_latent=gpt_cond_latent,
                speaker_embedding=speaker_embedding,
                temperature=0.3,
                length_penalty=1.0,
                repetition_penalty=10.0,
                top_k=30,
                top_p=0.85,
            )

            # Adjust length for short sentences
            keep_len = self._calculate_keep_len(chunk, lang_code)
            wav_chunk["wav"] = torch.tensor(wav_chunk["wav"][:keep_len])

            if output_chunks:
                chunk_path = os.path.join(self.output_dir, f"{self._get_file_name(chunk)}.wav")
                torchaudio.save(chunk_path, wav_chunk["wav"].unsqueeze(0), 24000)
                if verbose:
                    print(f"Saved chunk to {chunk_path}")

            wav_chunks.append(wav_chunk["wav"])

        # Combine all chunks and save
        final_wav = torch.cat(wav_chunks, dim=0).unsqueeze(0)
        output_path = os.path.join(self.output_dir, f"{self._get_file_name(text)}.wav")
        torchaudio.save(output_path, final_wav, 24000)

        if verbose:
            print(f"Saved final file to {output_path}")

        return output_path

# Example usage
# if __name__ == "__main__":
#     # Initialize the model
#     tts = VietnameseXTTS()
#     reference_audio = "2_20231227_1742037149.wav"
#     # Generate speech
#     output_file = tts.generate_speech(
#         text="Xin chào mọi người. Hôm nay là một ngày đẹp trời.",
#         speaker_audio_file=reference_audio,
#         language="Tiếng Việt",
#         normalize_text=True,
#         verbose=True,
#         output_chunks=True
#     )
#     print(f"Generated audio saved to: {output_file}")

In [16]:
tts = VietnameseXTTS()
reference_audio = "2_20231227_1742037149.wav"
# Generate speech
# output_file = tts.generate_speech(
#     text="Xin chào mọi người. Hôm nay là một ngày đẹp trời.",
#     speaker_audio_file=reference_audio,
#     language="Tiếng Việt",
#     normalize_text=True,
#     verbose=True,
#     output_chunks=True
# )


Loading XTTS model...
[2025-02-10 18:00:53,414] [INFO] [logging.py:128:log_dist] [Rank -1] DeepSpeed info: version=0.16.3, git-hash=unknown, git-branch=unknown
[2025-02-10 18:00:53,416] [WARNING] [config_utils.py:70:_process_deprecated_field] Config parameter replace_method is deprecated. This parameter is no longer needed, please remove from your call to DeepSpeed-inference
[2025-02-10 18:00:53,417] [WARNING] [config_utils.py:70:_process_deprecated_field] Config parameter mp_size is deprecated use tensor_parallel.tp_size instead
[2025-02-10 18:00:53,418] [INFO] [logging.py:128:log_dist] [Rank -1] quantize_bits = 8 mlp_extra_grouping = False, quantize_groups = 1
[2025-02-10 18:00:53,574] [INFO] [logging.py:128:log_dist] [Rank -1] DeepSpeed-Inference config: {'layer_id': 0, 'hidden_size': 1024, 'intermediate_size': 4096, 'heads': 16, 'num_hidden_layers': -1, 'dtype': torch.float32, 'pre_layer_norm': True, 'norm_type': <NormType.LayerNorm: 1>, 'local_rank': -1, 'stochastic_mode': False, 

In [17]:
input_text ="""Cùng dự, có các Phó Thủ tướng Chính phủ Nguyễn Hòa Bình, Trần Hồng Hà, Lê Thành Long và Bùi Thanh Sơn; lãnh đạo các bộ, ngành Trung ương, các hiệp hội doanh nghiệp, 26 doanh nghiệp nhà nước và tư nhân lớn.
Phát biểu ý kiến khai mạc hội nghị, thay mặt Tổng Bí thư Tô Lâm, các đồng chí Lãnh đạo Đảng, Nhà nước, Thủ tướng Phạm Minh Chính gửi tới các doanh nghiệp lời chào trân trọng, lời thăm hỏi ân cần, lời chúc mừng tốt đẹp nhất; nêu rõ, chúng ta đã bước sang năm cuối của nhiệm kỳ Đại hội lần thứ XIII của Đảng, đây là giai đoạn khó khăn với đại dịch Covid-19 hoành hành; chiến tranh, xung đột trên thế giới làm đứt gãy chuỗi cung ứng; cơn bão số 3 (Yagi) gây hậu quả nghiêm trọng; trong năm 2024 có sự thay đổi nhân sự lãnh đạo cấp cao, sự ra đi đột ngột của Tổng Bí thư Nguyễn Phú Trọng... tác động đến tình hình kinh tế-xã hội, an ninh chính trị, quốc phòng… đất nước nhưng dưới sự lãnh đạo của Đảng, thường xuyên trực tiếp là Bộ Chính trị, Ban Bí thư, đứng đầu là đồng chí Tổng Bí thư, cả hệ thống chính trị vào cuộc, có sự ủng hộ, đồng tình của người dân, doanh nghiệp, sự giúp đỡ của bạn bè quốc tế, chúng ta đã nỗ lực vượt qua mọi khó khăn khốc liệt, đạt được những thành tựu phát triển kinh tế-xã hội ấn tượng.
""" 

output_file = tts.generate_speech(
    text=input_text,
    speaker_audio_file=reference_audio,
    language="Tiếng Việt",
    normalize_text=True,
    verbose=True,
    output_chunks=True
)

Roman  thứ 13


Processing 5 chunks:
['Cùng dự  có các Phó Thủ tướng Chính phủ Nguyễn Hòa Bình  Trần Hồng Hà  Lê Thành Long và Bùi Thanh Sơn  lãnh đạo các bộ  ngành Trung ương  các hiệp hội doanh nghiệp  hai mươi sáu doanh nghiệp nhà nước và tư nhân lớn. Phát biểu ý kiến khai mạc hội nghị', ' các đồng chí Lãnh đạo Đảng  Nhà nước  Thủ tướng Phạm Minh Chính gửi tới các doanh nghiệp lời chào trân trọng  lời thăm hỏi ân cần  lời chúc mừng tốt đẹp nhất  nêu rõ  chúng ta đã bước sang năm cuối của nhiệm kỳ Đại hội lần thứ mười ba của Đảng', ' chiến tranh  xung đột trên thế giới làm đứt gãy chuỗi cung ứng  cơn bão số ba Yagi gây hậu quả nghiêm trọng  trong năm hai nghìn không trăm hai mươi tư có sự thay đổi nhân sự lãnh đạo cấp cao', ' an ninh chính trị  quốc phòng… đất nước nhưng dưới sự lãnh đạo của Đảng  thường xuyên trực tiếp là Bộ Chính trị  Ban Bí thư  đứng đầu là đồng chí Tổng Bí thư  cả hệ thống chính trị vào cuộc  có sự ủng hộ  đồng tình của người dân  doanh nghiệp', ' chúng ta đã nỗ lực vượt qua mọi

  0%|                                                                                                                                                                                              | 0/5 [00:00<?, ?it/s]

ASJHKAHSKJA 250


 20%|████████████████████████████████████▍                                                                                                                                                 | 1/5 [00:03<00:13,  3.27s/it]

...

Saved chunk to ./output/0210180102__chien_tranh__xung_dot_tren_the_gioi_lam_dut_gay_c.wav
ASJHKAHSKJA 243


 80%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                    | 4/5 [00:11<00:02,  2.78s/it]

Saved chunk to ./output/0210180105__an_ninh_chinh_tri__quoc_phong..._dat_nuoc_nhung_duo.wav
ASJHKAHSKJA 113


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:12<00:00,  2.56s/it]

Saved chunk to ./output/0210180106__chung_ta_da_no_luc_vuot_qua_moi_kho_khan_khoc_lie.wav
Saved final file to ./output/0210180106_cung_du_co_cac_pho_thu_tuong_chinh_phu_nguyen_hoa.wav


In [18]:
print(f"Generated audio saved to: {output_file}")

from IPython.display import Audio
Audio(output_file)

Generated audio saved to: ./output/0210180106_cung_du_co_cac_pho_thu_tuong_chinh_phu_nguyen_hoa.wav
